In [2]:
# load a .pt textual inversion file and show / manipulate it
import torch
import numpy as np
import os

1. load a PyTorch tensor from a TI file in the 'textual_inversions' directory. load onto the CPU.
2. print the keys, which is a dictionary, then print the key/value pairs.
3. extract the tensor with the key '*' from the dict under the 'string_to_param' key.
4. convert the tensor to a NumPy array and detach from the GPU. All subsequent ops on NumPy array.
5. show number of vectors and shape of tensor.

In [7]:
filename = 'TI_Tron_original' # exclude the .pt extension

data = torch.load(filename+'.pt', map_location='cpu')
print('TI loaded successfully from ',filename+'.pt')
print('-----------------------------')
print(data.keys())
print('-----------------------------')
#for each key, print the data
for key in data.keys():
    print('key: ',key,' data[key]: ',data[key])

# Extract the tensor associated with the key '*'
tensor = data['string_to_param']['*']

# Convert tensor to a NumPy array. detach tensor from the GPU
# all ops will be done on the numpy array
np_array = tensor.cpu().detach().numpy()
numvectors = np_array.shape[0] # number of vectors
print('\nNumber of vectors: ',numvectors,'    Shape of the tensor: ',np_array.shape)

#create a copy of the numpy array
np_rolled = np_array.copy()

TI loaded successfully from  TI_Tron_original.pt
-----------------------------
dict_keys(['string_to_token', 'string_to_param', 'name', 'step', 'sd_checkpoint', 'sd_checkpoint_name'])
-----------------------------
key:  string_to_token  data[key]:  {'*': 265}
key:  string_to_param  data[key]:  {'*': tensor([[ 0.0332,  0.0405,  0.0354,  ...,  0.0685, -0.0129, -0.0338],
        [-0.0588, -0.0188,  0.0036,  ...,  0.0679, -0.0458,  0.0542],
        [ 0.0784, -0.0162,  0.0400,  ...,  0.0390,  0.0686,  0.0756],
        ...,
        [ 0.0364, -0.0542,  0.0208,  ..., -0.0587,  0.0521,  0.0246],
        [ 0.0437, -0.0197, -0.0377,  ...,  0.0301, -0.0330,  0.0273],
        [-0.0385, -0.0180, -0.0414,  ..., -0.0428,  0.0292,  0.0398]],
       requires_grad=True)}
key:  name  data[key]:  Style-TronLegacy-8v-2280
key:  step  data[key]:  2279
key:  sd_checkpoint  data[key]:  81761151
key:  sd_checkpoint_name  data[key]:  v1-5-pruned-emaonly

Number of vectors:  8     Shape of the tensor:  (8, 768)


/tmp/ipykernel_12086/2789692199.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(filename+'.pt', map_location='cpu')


In [8]:
# ROLLING
#########

roll_amount = int(input("ROLLING: Enter eg '3' (can be +ve or -ve) : "))

#roll_amount *each row* of the numpy array *separately*
for i in range(numvectors):
    np_rolled[i] = np.roll(np_rolled[i], roll_amount)

# SAVING
########

tensor = torch.tensor(np_rolled, device='cuda:0', requires_grad=True) 
print(tensor.shape)
data['string_to_param']['*'] = tensor  # store the tensor back in the data dictionary

# Add prefix for saving rolled file
filename = filename + "_roll" + str(roll_amount) + ".pt"

# Construct the file path
filepath = os.path.join(os.getcwd(), filename)

# Save file
torch.save(data, filepath)
print(f"The file '{filename}' has been saved successfully.")


torch.Size([8, 768])
The file 'TI_Tron_original_roll-1.pt' has been saved successfully.
